# Phase 2: Legal NER - Encoder-Decoder, Full Fine-tuning

## Task
Legal named entity recognition (NER): find entities (names, courts, statutes, dates and so on) in legal judgments. Every word gets a B/I/O tag, and predictions are scored with strict macro F1 (start, end and label must all match).

Model: the encoder of the pre-trained encoder-decoder with a tag layer on top. All parameters are trained.

## Input
- Phase-2 NER data (needs the HF_TOKEN secret).
- Phase-1 files legal_encdec.pt and tok_encdec.json.

## Flow of the notebook (from input to output)
1. Write the shared helpers: data access, encoder-decoder model, training loop, submission writer.
2. Set the settings and load the pre-trained model.
3. Load the documents and turn the character entity spans into B/I/O tags on tokens, using overlapping windows.
4. Fine-tune the encoder with a tag layer and check the strict macro F1 on dev.
5. Predict the test documents (averaging overlapping windows) and write the submission zip.

## Output
- Submission zip with the test entities.
- One zip with all outputs.

In [1]:
# Shared code: data, encoder-decoder model, training loop
import os, glob, json, math, time, random, zipfile, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer
DEV, PAD, BOS, EOS, SEP = "cuda" if torch.cuda.is_available() else "cpu", 0, 1, 2, 3
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./out"; os.makedirs(OUT, exist_ok=True)
# It takes rel (file path in the dataset) as input, searches for a file and returns its path.
def find(rel): h = glob.glob(f"./**/{rel}", recursive=True) + glob.glob(f"/kaggle/input/**/{rel}", recursive=True); return h[0] if h else None
# It takes rel (file path in the dataset) as input, gets a dataset file: local copy if present, otherwise download it and returns a local file path.
def getf(rel):
    from huggingface_hub import hf_hub_download
    try: from kaggle_secrets import UserSecretsClient; t = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception: t = os.environ.get("HF_TOKEN", "")
    return find(rel) or hf_hub_download("Exploration-Lab/CS-781-Capstone", rel, repo_type="dataset", token=t)
jload = lambda rel: json.load(open(getf(rel), encoding="utf-8"))
# It takes rel (file path in the dataset) as input, loads a JSON file as a list of records and returns a list of records.
def rec(rel):
    d = jload(rel)
    if isinstance(d, dict) and isinstance(d.get("text"), list): return [d]
    return [{**v, "id": v.get("id", k)} if isinstance(v, dict) else {"id": k, "text": v} for k, v in d.items()] if isinstance(d, dict) else d
class Attn(nn.Module):
    # It takes d (model size) and h (number of heads) as input, creates the attention layer and returns nothing.
    def __init__(s, d, h): super().__init__(); s.h = h; s.q = nn.Linear(d, d); s.kv = nn.Linear(d, 2 * d); s.o = nn.Linear(d, d)
    # It takes x (input tensor), mem (encoder output), mask (attention mask) and causal (use causal mask) as input, runs self-attention or cross-attention and returns output vectors.
    def forward(s, x, mem=None, mask=None, causal=False):
        m = x if mem is None else mem; B, T, D = x.shape; q = s.q(x).view(B, T, s.h, -1).transpose(1, 2); k, v = s.kv(m).view(B, m.size(1), 2, s.h, -1).permute(2, 0, 3, 1, 4)
        return s.o(F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=causal and mask is None).transpose(1, 2).reshape(B, T, D))
class Block(nn.Module):
    # It takes d (model size), h (number of heads), cross (add cross-attention) and p (dropout rate) as input, creates one Transformer block and returns nothing.
    def __init__(s, d, h, cross=False, p=0.1):
        super().__init__(); s.n1, s.sa, s.cross = nn.LayerNorm(d), Attn(d, h), cross
        if cross: s.nx, s.ca = nn.LayerNorm(d), Attn(d, h)
        s.n2, s.ff, s.dp = nn.LayerNorm(d), nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d)), nn.Dropout(p)
    # It takes x (input tensor), mem (encoder output), mm (encoder padding mask), causal (use causal mask) and mask (attention mask) as input, runs one Transformer block and returns output vectors.
    def forward(s, x, mem=None, mm=None, causal=False, mask=None):
        x = x + s.dp(s.sa(s.n1(x), mask=mask, causal=causal))
        if s.cross and mem is not None: x = x + s.dp(s.ca(s.nx(x), mem, mm))
        return x + s.dp(s.ff(s.n2(x)))
class EncDec(nn.Module):
    # It takes V (vocabulary size), d (model size), h (number of heads), ne (encoder layers), nd (decoder layers), L (max length) and p (dropout rate) as input, builds the encoder-decoder model and returns nothing.
    def __init__(s, V=16000, d=640, h=10, ne=6, nd=8, L=1024, p=0.1):
        super().__init__(); s.emb, s.pos = nn.Embedding(V, d), nn.Embedding(L, d); s.enc = nn.ModuleList(Block(d, h, p=p) for _ in range(ne)); s.dec = nn.ModuleList(Block(d, h, True, p) for _ in range(nd)); s.ne_, s.nd_ = nn.LayerNorm(d), nn.LayerNorm(d)
    # It takes x (input tensor) as input, turns token ids into vectors with positions and returns embedding vectors.
    def embed(s, x): return s.emb(x) + s.pos(torch.arange(x.size(1), device=x.device))
    # It takes src (source token ids) as input, runs the encoder over the input text and returns (encoder output, padding mask).
    def encode(s, src):
        mm = (src != PAD)[:, None, None, :]; x = s.embed(src)
        for b in s.enc: x = b(x, mask=mm)
        return s.ne_(x), mm
    # It takes tgt (target token ids), mem (encoder output) and mm (encoder padding mask) as input, runs the decoder and returns next-token scores.
    def decode(s, tgt, mem=None, mm=None):
        x = s.embed(tgt)
        for b in s.dec: x = b(x, mem, mm, causal=True)
        return F.linear(s.nd_(x), s.emb.weight)
    # It takes src (source token ids) and tgt (target token ids) as input, runs encoder then decoder and returns next-token scores (logits).
    def forward(s, src, tgt): return s.decode(tgt, *s.encode(src))
# It loads the pre-trained model, its settings and tokenizer and returns (model, settings, tokenizer).
def base():
    ck = torch.load(find("legal_encdec.pt"), map_location="cpu"); m = EncDec(**ck["cfg"]); m.load_state_dict(ck["sd"]); return m, ck["cfg"], Tokenizer.from_file(find("tok_encdec.json"))
# It takes rows (token lists) and T (length to pad to) as input, pads token lists to the same length and returns a padded tensor of token ids.
def padto(rows, T=None):
    T = T or max(len(r) for r in rows); a = np.full((len(rows), T), PAD, np.int64)
    for i, r in enumerate(rows): a[i, :len(r)] = r[:T]
    return torch.from_numpy(a)
# It takes m (model), ids (token ids) and w (token weights) as input, averages the encoder outputs over the real tokens and returns one vector per row.
def pool(m, ids, w=None):
    h, mm = m.encode(ids); w = (mm[:, 0, 0, :] if w is None else w).to(h.dtype)[..., None]; return (h * w).sum(1) / w.sum(1).clamp(min=1)
# It takes m (model), loss_fn (loss function), sample (batch function), steps (number of steps), lr (learning rate), ev (evaluation function), every (how often), hours (time limit in hours), acc (gradient accumulation steps), hm (hours margin), warm (warm-up steps), log (print function), fill (progress fill) and patience as input, trains the model with warm-up, cosine learning rate and validation checks and returns training history and the best weights.
def fit(m, loss_fn, sample, steps, lr, ev, every, hours, acc=1, hm=1, warm=100, log=50, fill=False, patience=0):
    m.to(DEV).train(); ps = [p for p in m.parameters() if p.requires_grad]
    head = [p for n, p in m.named_parameters() if n.startswith("head")]; rest = [p for p in ps if all(p is not h for h in head)]
    opt = torch.optim.AdamW([dict(params=g, mult=k) for g, k in ((rest, 1), (head, hm)) if g], lr=lr, weight_decay=0.01, betas=(0.9, 0.95))
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda"); t0, best, best_state, run, s, bad = time.time(), 1e9, None, 0.0, 0, 0
    while s < steps:
        if s == 25: budget = max(30, int(hours * 3600 / ((time.time() - t0) / 25))); steps = budget if fill else min(steps, budget)
        decay = 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * s / steps))
        for g in opt.param_groups: g["lr"] = lr * g["mult"] * min(1, (s + 1) / warm) * decay
        for _ in range(acc):
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"): loss = loss_fn(m, *sample()) / acc
            scaler.scale(loss).backward(); run += loss.item()
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(ps, 1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); s += 1
        if s % log == 0: print(f"step {s} loss {run / log:.4f}", flush=True); run = 0.0
        if s % every == 0 or s == steps:
            v = ev(m); m.train(); print(f"   eval {v:.4f}", flush=True)
            bad = 0 if v < best else bad + 1
            if v < best: best, best_state = v, {k: p.detach().cpu().clone() for k, p in m.named_parameters() if p.requires_grad}
            if patience and bad >= patience: break
    if best_state: m.load_state_dict(best_state, strict=False)
    return m.eval()
# It takes obj (object) and tag (label) as input, writes predictions into a submission zip and returns a path of the submission zip.
def submit(obj, tag):
    p = f"{OUT}/{tag}_prediction.json"; json.dump(obj, open(p, "w"), ensure_ascii=False); z = f"{OUT}/{tag}_submission.zip"
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as f: f.write(p, "prediction.json")
    return z
# It takes name as input, zips all output files into one archive and returns a path of the zip file.
def zip_all(name="ALL_outputs.zip"):
    with zipfile.ZipFile(f"{OUT}/{name}", "w", zipfile.ZIP_DEFLATED) as f:
        for p in sorted(glob.glob(f"{OUT}/*")):
            if os.path.isfile(p) and not p.endswith(name) and not p.endswith(".npy"): f.write(p, os.path.basename(p))
    print(f"{OUT}/{name}", f"{os.path.getsize(f'{OUT}/{name}') / 1e6:.1f} MB")

In [2]:
# Setup: settings and load the pre-trained model
SMOKE = False
WIN, STRIDE, BS, EPOCHS, HOURS, LR, HM = 510, 384, 16, 30, 4.5, 1e-4, 10
if SMOKE: WIN, STRIDE, BS, EPOCHS = 128, 96, 4, 1
m, cfg, tok = base()

In [3]:
# Load data and turn entity spans into B/I/O tags
labs = [l.strip() for l in open(getf("Phase_2_LNER/Train/labels.txt")) if l.strip()]
TAGS = ["O"] + [f"{p}-{l}" for l in labs for p in "BI"]; T2I = {t: i for i, t in enumerate(TAGS)}
# It takes g (annotations) as input, puts the entity annotations into one format and returns a dictionary from document id to entities.
def norm(g): return g if isinstance(g, dict) else {str(r["id"]): r.get("entities") or r.get("annotations") or [] for r in g}
# It takes g (annotations) as input, reads entities as (start, end, label) tuples and returns a list of (start, end, label).
def ents(g): return [(int(e[0]), int(e[1]), e[2]) if isinstance(e, (list, tuple)) else (int(e["start"]), int(e["end"]), e["label"]) for e in g]
text = {**jload("Phase_2_LNER/Train/fold1-text.json"), **jload("Phase_2_LNER/Train/fold2-text.json")}
gold = {**norm(jload("Phase_2_LNER/Train/fold1-gold.json")), **norm(jload("Phase_2_LNER/Train/fold2-gold.json"))}
D = {}
for k, t in text.items():
    e = tok.encode(t); st, en = np.array([o[0] for o in e.offsets]), np.array([o[1] for o in e.offsets]); y = np.zeros(len(e.ids), np.int64)
    for s, f, l in sorted(ents(gold.get(k, []))):
        ix = np.nonzero((st < f) & (en > s))[0]; y[ix] = [T2I[("B-" if j == 0 else "I-") + l] for j in range(len(ix))]
    D[k] = (e.ids, e.offsets, y)
keys = sorted(D)
# It takes n (number) as input, starts positions of overlapping windows and returns a list of start positions.
def wins(n): return list(range(0, max(n - WIN, 0) + 1, STRIDE)) + ([n - WIN] if n > WIN and (n - WIN) % STRIDE else [])
print(len(keys), "documents")

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

70 documents


In [ ]:
# Fine-tune the model for NER and check dev score
m.head = nn.Linear(cfg["d"], len(TAGS)); wt = torch.ones(len(TAGS)); wt[0] = 0.3
rng = np.random.default_rng(0)
# It picks a random batch of training examples and returns one training batch.
def sample():
    b = [items[i] for i in rng.integers(0, len(items), BS)]
    return padto([D[k][0][w:w + WIN] for k, w in b]).to(DEV), padto([D[k][2][w:w + WIN] + 1 for k, w in b]).to(DEV)
# It takes m (model), x (input ids) and y (labels) as input, computes the training loss for one batch and returns a loss value.
def lossf(m, x, y): return F.cross_entropy(m.head(m.encode(x)[0]).float().reshape(-1, len(TAGS)), (y - 1).reshape(-1), weight=wt.to(DEV), ignore_index=-1, label_smoothing=0.1)
# It takes t (document text), offs (character offsets) and tags as input, turns predicted tags into entity spans and returns a list of entity spans.
def decode(t, offs, tags):
    out, cur = [], None
    for (a, b), g in zip(offs, tags):
        p, l = ("O", "") if g == 0 else TAGS[g].split("-", 1)
        if p == "O":
            if cur: out.append(cur); cur = None
        elif p == "B" or cur is None or cur[2] != l:
            if cur: out.append(cur)
            cur = [a, b, l]
        else: cur[1] = b
    if cur: out.append(cur)
    res = []
    for a, b, l in out:
        while a < b and t[a].isspace(): a += 1
        while b > a and t[b - 1].isspace(): b -= 1
        if b > a: res.append([a, b, l, t[a:b]])
    return res
# It takes ids (token ids of one document) as input, gets token probabilities for one document and returns token probabilities.
@torch.no_grad()
def probs(ids):
    P = np.zeros((len(ids), len(TAGS)), np.float32); C = np.zeros(len(ids), np.float32); ws = wins(len(ids))
    for i in range(0, len(ws), 16):
        b = ws[i:i + 16]; x = padto([ids[w:w + WIN] for w in b]).to(DEV)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"): pr = F.softmax(m.head(m.encode(x)[0]).float(), -1).cpu().numpy()
        for r, w in enumerate(b): k2 = len(ids[w:w + WIN]); P[w:w + k2] += pr[r, :k2]; C[w:w + k2] += 1
    return P / np.maximum(C, 1)[:, None]
# It takes texts and sds (model weights) as input, averages probabilities from several models and returns probabilities (or spans) per document.
def predict(texts, sds=(None,)):
    enc = {k: tok.encode(t) for k, t in texts.items()}; S = {}
    for sd in sds:
        if sd is not None: m.load_state_dict(sd, strict=False)
        m.eval()
        for k, e in enc.items():
            if e.ids: S[k] = S.get(k, 0) + probs(e.ids)
    return {k: decode(texts[k], e.offsets, S[k].argmax(1)) if e.ids else [] for k, e in enc.items()}
# It takes pred (predictions) and ref (reference text) as input, computes strict macro F1 for entity spans and returns macro F1.
def strict_f1(pred, ref):
    f = []
    for l in [l for l in labs if any(e[2] == l for v in ref.values() for e in v)]:
        P = {(d, s, e) for d, v in pred.items() for s, e, ll, *_ in v if ll == l}; G = {(d, s, e) for d, v in ref.items() for s, e, ll, *_ in v if ll == l}
        tp = len(P & G); p, r = tp / max(len(P), 1), tp / max(len(G), 1); f.append(2 * p * r / max(p + r, 1e-9))
    return float(np.mean(f))
ref = {k: [list(e) for e in ents(gold.get(k, []))] for k in keys}
print("tokenisation upper bound (gold tags -> spans, strict F1):", round(strict_f1({k: decode(text[k], D[k][1], D[k][2]) for k in keys}, ref), 4))
LRS, MAXMIN = [5e-5, 3e-5, 1e-4], 25
start, MS, scores, t0 = {k: v.clone() for k, v in m.state_dict().items()}, [], [], time.time()
while time.time() - t0 < HOURS * 3600 - MAXMIN * 60 and len(MS) < (2 if SMOKE else 99):
    seed = len(MS); LR = LRS[seed % 3]; torch.manual_seed(seed); rng = np.random.default_rng(seed)
    ks = keys[:]; random.Random(seed).shuffle(ks); nd = max(2, len(ks) // 8); dev, trn = ks[:nd], ks[nd:]
    items = [(k, w) for k in trn for w in wins(len(D[k][0]))]
    m.load_state_dict(start, strict=False); m.head = nn.Linear(cfg["d"], len(TAGS))
    fit(m, lossf, sample, 6000 if not SMOKE else 20, LR, lambda m: -strict_f1(predict({k: text[k] for k in dev}), {k: ref[k] for k in dev}), 100 if not SMOKE else 10, MAXMIN / 60, hm=HM, patience=6)
    f1 = strict_f1(predict({k: text[k] for k in dev}), {k: ref[k] for k in dev}); scores.append(f1)
    MS.append({k: v.detach().cpu().half() for k, v in m.state_dict().items()}); print(f"model {len(MS)}: lr {LR}, dev strict macro-F1 {f1:.4f}  (mean so far {np.mean(scores):.4f})")
f1 = float(np.mean(scores)); print(len(MS), "models; mean single-model dev strict macro-F1:", round(f1, 4))

tokenisation upper bound (gold tags -> spans, strict F1): 0.9869
step 50 loss 3.6782
step 100 loss 2.8729
   eval -0.0000
step 150 loss 2.2859
step 200 loss 2.1972
   eval -0.0070
step 250 loss 2.1427
step 300 loss 2.0741
   eval -0.0124
step 350 loss 2.0149
step 400 loss 1.9597
   eval -0.0310
step 450 loss 1.9271
step 500 loss 1.8944
   eval -0.0525
step 550 loss 1.8684
step 600 loss 1.8425
   eval -0.0638
step 650 loss 1.8115
step 700 loss 1.8006
   eval -0.0695
step 750 loss 1.7765
step 800 loss 1.7635
   eval -0.0697
step 850 loss 1.7511
step 900 loss 1.7276
   eval -0.0644
step 950 loss 1.7156
step 1000 loss 1.7070
   eval -0.0680
step 1050 loss 1.6920
step 1100 loss 1.6826
   eval -0.0606
step 1150 loss 1.6650
step 1200 loss 1.6641
   eval -0.0605
step 1250 loss 1.6479
step 1300 loss 1.6414
   eval -0.0621
step 1350 loss 1.6327
step 1400 loss 1.6280
   eval -0.0630
model 1: lr 5e-05, dev strict macro-F1 0.0697  (mean so far 0.0697)
step 50 loss 3.7253
step 100 loss 3.2490
   eva

In [ ]:
# Predict on test set and make submission zip
print(submit(predict(jload("Phase_2_LNER/Test/fold3-text.json"), MS), f"p2_full_dev{f1:.4f}")); zip_all()